# Step 3 - Modèle de prédiction des retards (Random Forest)

Objectif :
Construire un modèle de régression qui peut prédire le retard moyen
à l’arrivée à partir des caractéristiques du trajet.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [2]:
# Chargement des données nettoyées
df = pd.read_csv("cleaned_dataset.csv")

# suppression index inutile
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df.head()

,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,...,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)",year,month,day
0,2018-01-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870,5,289,11.247809,3.693179,...,8,36.134454,31.092437,10.924370,15.966387,5.040000,31.092437,2018.0,1.0,1.0
1,2018-01-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406,1,213,8.479969,4.567119,...,4,20.000000,35.000000,16.666667,16.666667,8.333333,35.000000,2018.0,1.0,1.0
2,2018-01-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226,0,21,6.239683,0.286283,...,1,22.222222,27.777778,16.666667,16.666667,5.555556,27.777778,2018.0,1.0,1.0
3,2018-01-01,National,PARIS MONTPARNASSE,NANTES,216.21,508,3,71,7.235211,0.980000,...,8,33.333333,22.222222,16.666667,20.370370,5.555556,22.222222,2018.0,1.0,1.0
4,2018-01-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472,4,224,6.784673,3.229701,...,0,15.789474,45.614035,8.333333,15.789474,1.754386,45.614035,2018.0,1.0,1.0


In [3]:
# Variable à prédire
target = "Retard moyen de tous les trains à l'arrivée"

y = df[target]

In [4]:
#On garde pas les variables directement liées au retard final.
features = df.drop(columns=[
    target,
    "Retard moyen des trains en retard à l'arrivée",
    "Retard moyen de tous les trains au départ"
], errors="ignore")

X = features

In [5]:
# on identifie les types de variables

categorical_cols = X.select_dtypes(include="object").columns
numeric_cols = X.select_dtypes(include=np.number).columns

categorical_cols, numeric_cols

(Index(['Date', 'Service', 'Gare de départ', 'Gare d'arrivée'], dtype='object'),
 Index(['Durée moyenne du trajet', 'Nombre de circulations prévues',
        'Nombre de trains annulés', 'Nombre de trains en retard au départ',
        'Retard moyen des trains en retard au départ',
        'Nombre de trains en retard à l'arrivée',
        'Nombre trains en retard > 15min',
        'Retard moyen trains en retard > 15 (si liaison concurrencée par vol)',
        'Nombre trains en retard > 30min', 'Nombre trains en retard > 60min',
        'Prct retard pour causes externes',
        'Prct retard pour cause infrastructure',
        'Prct retard pour cause gestion trafic',
        'Prct retard pour cause matériel roulant',
        'Prct retard pour cause gestion en gare et réutilisation de matériel',
        'Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)',
        'year', 'month', 'day'],
       dtype='object'))

In [6]:
# Préprocessing complet :
# - Remplacer les NaN numériques par la médiane
# - Remplacer les NaN catégoriels par la valeur la plus fréquente
# - Encoder les catégories

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

In [7]:
# Création du modèle

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [8]:
# Pipeline = preprocessing + modèle

pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", model)
])

In [9]:
# Séparation entraînement / test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
# Diagnostic : vérifier NaN dans X et dans y
print("NaN dans y :", y.isna().sum())

print("NaN dans X (total) :", X.isna().sum().sum())

# Colonnes avec NaN
nan_cols = X.columns[X.isna().any()].tolist()
print("Colonnes avec NaN :", nan_cols[:20], "..." if len(nan_cols) > 20 else "")
print("Nombre de colonnes avec NaN :", len(nan_cols))

NaN dans y : 0
NaN dans X (total) : 295
Colonnes avec NaN : ['Gare de départ', "Gare d'arrivée", 'year', 'month', 'day'] 
Nombre de colonnes avec NaN : 5


In [11]:
# Entraînement du modèle

pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['Durée moyenne du trajet', 'Nombre de circulations prévues',
       'Nombre de trains annulés', 'Nombre de trains en retard au départ',
       'Retard moyen des trains en retard au départ',
       'Nombre de trains en retard à l'arrivée',
       'Nombre trains en retard > 15...
       'Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)',
       'year', 'month', 'day'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Date', 'Service', 'Gare de départ', 'Gare d'arrivée'], dtype='object'))])),
                ('model', RandomForestRegressor(n_jobs=-1, random_state=42))])

In [12]:
# Prédictions

y_pred = pipeline.predict(X_test)

In [16]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

MAE : 1.0494656917845944
RMSE : 2.186751899835644
R² : 0.7711365154679357


### Interprétation des performances du modèle Random Forest

Les performances du modèle ont été évaluées à l’aide de trois métriques de régression : MAE, RMSE et R².

Le modèle obtient une erreur absolue moyenne (MAE) d’environ 1,05 minute, ce qui signifie que les prédictions du modèle s’écartent en moyenne d’un peu plus d’une minute par rapport aux retards réels.

Le RMSE est d’environ 2,19 minutes, ce qui montre que les erreurs plus importantes restent limitées et que le modèle gère correctement la majorité des situations, même en présence de retards plus élevés.

Le coefficient de détermination R² atteint une valeur de 0,77, ce qui signifie que le modèle explique environ 77 % de la variabilité des retards observés. Cela indique une forte capacité prédictive et montre que les variables utilisées contiennent une information pertinente pour estimer les retards ferroviaires.

Globalement, ces résultats démontrent que le modèle Random Forest est capable de prédire efficacement les retards moyens à l’arrivée et constitue une amélioration significative par rapport à une prédiction naïve basée uniquement sur la moyenne.